# Annotating the Ammonium Transporter family tree

The Python package `ete3` allows for programmatic annotation and visualisation, which can be a bit of a time-saver.

In [6]:
# Python imports
from pathlib import Path
import ete3
from ete3 import Tree, TreeStyle, NodeStyle, TextFace, CircleFace, faces
import pandas as pd
# Essential imports
import re

from io import StringIO

import pandas as pd
import numpy as np

from Bio import SeqIO
from bioservices import UniProt
from bioservices import Pfam

import tqdm

# Connect to UniProt service
u = UniProt(verbose=False)

In [26]:
# File locations and other constants
treepath = Path("QTroot.new")  # transporter family tree, Newick format
annopath = Path("all_annotations.csv")     # annotations for some family members


We load in the tree and annotation data

In [27]:
# Load the tree and, for convenience, root it to the midpoint
# We have these as functions so that we can annotate/render a clean tree each time
def load_tree():
    tree = Tree(str(treepath))
    # midpoint = tree.get_midpoint_outgroup()  # Calculate the midpoint node
    # tree.set_outgroup(midpoint)# and set it as tree outgroup
    # tree.ladderize()
    return tree

# Load the annotation
def load_annotation():
    anno = pd.read_csv(annopath)
    return anno


Now we have the data loaded, we can start to annotate the tree. First, we associate each leaf from the tree with its row in the annotation dataframe. Before the association, the dataframe looks like this:

In [28]:
load_annotation()  # Load the unaltered annotation as it is in the file

,Unnamed: 0,0,four,first,PheGate,TwinHis,SN,kingdom,Phyla,Class,EDG,Function,Sort
0,0,A0A858ZNJ3,YNSVAV,Y,FF,HH,S,Bacteria,Pseudomonadota,Betaproteobacteria,E,NaN,NaN
1,1,A0A1H9M798,YNNTAV,Y,FF,HH,S,Bacteria,Pseudomonadota,Betaproteobacteria,E,NaN,NaN
2,2,B0DY50,YNSMMV,Y,FF,HH,S,Eukaryota,Basidiomycota,Agaricomycetes,E,NaN,NaN
3,3,P69680,YNSMGV,Y,FF,HH,S,Bacteria,Pseudomonadota,Gammaproteobacteria,E,NaN,NaN
4,4,A0A3G9G4Q4,YNATAT,Y,FF,HH,S,Bacteria,Pseudomonadota,Alphaproteobacteria,E,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1276,1276,A0LV78,YNAMAT,Y,FF,HH,S,Bacteria,Actinomycetota,Actinomycetes,D,NaN,NaN
1277,1277,K5DBA0,ENDLTS,E,FF,HH,N,Bacteria,Planctomycetota,Planctomycetia,G,NaN,NaN
1278,1278,B8IZX4,YRSTAS,Y,FF,HH,S,Bacteria,Thermodesulfobacteriota,Desulfovibrionia,E,NaN,NaN
1279,1279,A8LKQ0,ENLYSS,E,FF,HH,N,Bacteria,Pseudomonadota,Alphaproteobacteria,E,NaN,NaN


We're going to add a new column with a reference to the corresponding leaf node of the tree. This isn't a particularly efficient way to go about things (a nested loop) but the dataset isn't that large.

In [29]:
# The function lets us return the tree and its corresponding annotation,
# where the leaves of this specific tree object are present in the
# annotation dataframe, and so can be manipulated easily.
def annotate_tree():
    tree = load_tree()
    anno = load_annotation()
    
    leaves = []  # Will hold leaves for the tree where they match the annotation row, or None if there is none
    
    for id in anno["0"]:               # iterate over annotations
        for leaf in tree.iter_leaves():  # iterate over all leaves in the tree
            assigned = False
            if id in str(leaf):
                leaves.append(leaf)
                assigned = True
                break
        if not assigned:
            leaves.append(None)
    
    anno["leaves"] = leaves

    return tree, anno

After the matching, the dataframe has a new column containing direct references to leaves on the tree:

In [30]:
tree, anno = annotate_tree()  # get the annotated tree
anno

,Unnamed: 0,0,four,first,PheGate,TwinHis,SN,kingdom,Phyla,Class,EDG,Function,Sort,leaves
0,0,A0A858ZNJ3,YNSVAV,Y,FF,HH,S,Bacteria,Pseudomonadota,Betaproteobacteria,E,NaN,NaN,(((\n--Alicycliphilus denitrificans Ammonium t...
1,1,A0A1H9M798,YNNTAV,Y,FF,HH,S,Bacteria,Pseudomonadota,Betaproteobacteria,E,NaN,NaN,(((\n--Giesbergeria anulus Ammonium transporte...
2,2,B0DY50,YNSMMV,Y,FF,HH,S,Eukaryota,Basidiomycota,Agaricomycetes,E,NaN,NaN,(((\n--Laccaria bicolor _strain S238N-H82 / AT...
3,3,P69680,YNSMGV,Y,FF,HH,S,Bacteria,Pseudomonadota,Gammaproteobacteria,E,NaN,NaN,(((\n--Escherichia coli O157_H7 Ammonium trans...
4,4,A0A3G9G4Q4,YNATAT,Y,FF,HH,S,Bacteria,Pseudomonadota,Alphaproteobacteria,E,NaN,NaN,(((\n--Asticcacaulis excentricus Ammonium tran...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1276,1276,A0LV78,YNAMAT,Y,FF,HH,S,Bacteria,Actinomycetota,Actinomycetes,D,NaN,NaN,(((\n--Acidothermus cellulolyticus _strain ATC...
1277,1277,K5DBA0,ENDLTS,E,FF,HH,N,Bacteria,Planctomycetota,Planctomycetia,G,NaN,NaN,(((\n--Rhodopirellula baltica SH28 Ammonium tr...
1278,1278,B8IZX4,YRSTAS,Y,FF,HH,S,Bacteria,Thermodesulfobacteriota,Desulfovibrionia,E,NaN,NaN,(((\n--Desulfovibrio desulfuricans _strain ATC...
1279,1279,A8LKQ0,ENLYSS,E,FF,HH,N,Bacteria,Pseudomonadota,Alphaproteobacteria,E,NaN,NaN,(((\n--Dinoroseobacter shibae _strain DSM 1649...


In [12]:
new_size = 4

def make_branches_bigger(node):
    node.img_style["hz_line_width"] = new_size # Change the horizotal lines stroke size
    node.img_style["vt_line_width"] = new_size # Change the vertical lines stroke size
    for c in node.children:
        make_branches_bigger(c)

This means we can select rows from within the dataframe, and address leaf nodes directly for visualisation.

## Visualising the annotated tree

With the data structure above, it's a little easier to use `ete3` to generate tree visualisations with arbitrarily annotated leaves

We can get a baseline visualisation of a semi circular tree with no labels, to get our bearings.

Now we can start to annotate the leaf nodes. We set a solid background colour first.

In [36]:
tree, anno = annotate_tree()  # get clean tree

# Declare tree style
everything.layout_fn = make_branches_bigger
everything = TreeStyle()
everything.show_leaf_name = True
everything.mode = "c"


# Set colours for residue types
rescolours = {"Y": "#F0E68C", "E": "#9ACD32", "M": "#FFA07A", "C": "#ADD8E6"}



# Iterate over leaves
for idx, row in anno.iterrows():
    # Get leaf information
    leaf = row["leaves"]
    restype = row["first"]
    # function = row["Function"]    
    kingdomname = row["kingdom"].strip()    

    # Style the leaf node
    if restype in ("Y", "E", "M", "C"):
        leaf.img_style["bgcolor"] = rescolours[restype]    
    # leaf.add_face(face_dict[kingdomname], 1, "aligned")

tree.render("figure_3.14.pdf", tree_style=everything, w=24, h=24, units="in");


In [34]:
tree, anno = annotate_tree()  # get clean tree

# Declare tree style
everything = TreeStyle()
everything.show_leaf_name = True
everything.mode = "c"
everything.show_branch_support = True

face_dict = {"Eukaryota": TextFace("eukaryota"),
             "Bacteria": TextFace("bacteria"),
             "Archaea": TextFace("archaea")}

# Set face colours for kingdoms
colour_dict = {"Eukaryota": "#FFFACD", "Bacteria": "#F0F8FF", "Archaea": "#FFE4E1"}

# Set colours for residue types
rescolours = {"Y": "#FFCC66", "E": "#009933", "M": "#9966FF", "C": "#1281280"}

# Iterate over leaves
for idx, row in anno.iterrows():
    # Get leaf information
    leaf = row["leaves"]
    restype = row["first"]
    function = row["Function"]    
    kingdomname = row["kingdom"].strip()    

    # Style the leaf node
    leaf.img_style["bgcolor"] = colour_dict[kingdomname]    
    leaf.add_face(face_dict[kingdomname], 1, "aligned")

    # Add gateway residue bubble
    if restype in ("Y", "E", "M", "C"):
        face = CircleFace(radius=20, color=rescolours[restype], style="sphere", label=restype)
        face.opacity = 0.3
        leaf.add_face(face, 1, position="float")

  
tree.render("figure_3.16.pdf", tree_style=everything, w=24, h=24, units="in");